# Week 2 · Your Week 1 code reviewed + daily recap quizzes

You asked for recap quizzes because the material doesn't stick yet. This notebook gives you that: **do one recap set at the start of each study day, closed book, 10 minutes.** Answer out loud or on paper first, then open the answer. Pulling something out of memory is what makes it stay, much more than re-reading it. Each set mixes old weeks with recent days on purpose, so you meet each idea again after a gap. First comes the review of your Week 1 code, which answers the questions you wrote in your cells.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## A. Your Week 1 code, reviewed

I read every cell you wrote in Week 1. All 18 exercises pass, and several of your comments are exactly right. Here are the things to take forward, starting with the most important.

**1. `super().__init__(...)` returns `None`, so `self.side` is `None`.** In `01_classes_lesson` and in exercise 3 you wrote `self.side = super().__init__(side, side)`. `area()` still works, because the parent's `__init__` sets `self.width` and `self.height`. But `__init__` only *sets up* the object and gives back nothing, so the value stored in `self.side` is `None`:

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height
    def area(self):
        return self.width * self.height

class SquareYours(Rectangle):
    def __init__(self, side):
        self.side = super().__init__(side, side)   # stores the RETURN value of __init__

class Square(Rectangle):
    def __init__(self, side):
        super().__init__(side, side)               # run the parent's setup
        self.side = side                           # then store side yourself, if you want it

print(SquareYours(5).area(), SquareYours(5).side)  # → 25 None
print(Square(5).area(), Square(5).side)            # → 25 5

**What happens, line by line, when you write `SquareYours(5)`**

```text
SquareYours(5)
 │
 ├─ 1. Python makes an empty object, call it obj
 ├─ 2. runs SquareYours.__init__(obj, side=5)
 │      │
 │      └─ self.side = super().__init__(5, 5)
 │                     │
 │                     ├─ 3. runs Rectangle.__init__(obj, 5, 5)
 │                     │       obj.width  = 5
 │                     │       obj.height = 5
 │                     │       (no return line → returns None)
 │                     │
 │                     └─ 4. the call's value is None
 │
 └─ 5. self.side = None        ← what got stored
```

- Step 3 did the useful work: `width` and `height` are set, which is why `area()` still gives $5 \times 5 = 25$.
- Step 4: every function returns *something*. With no `return` line, it returns `None`.
- Step 5: the `=` stores that return value. So `self.side` is `None`, not 5.

In plain words: **`super().__init__(...)` is an action, not a value.** Call it on its own line, then set your own attributes.

It's the same thing you worked out yourself in the `Stack.push` comment: `append` changes the list and returns `None`, so `return self.items.append(x)` hands back `None`. **Methods that change something in place (`append`, `sort`, `__init__`) return `None`.**

**2. Your `common_keys` passed by luck.** The task asked for a *sorted* list. Your loop returns the keys in `d1`'s order, which happened to be alphabetical in the test:

In [ ]:
def common_keys_yours(d1, d2):
    lst = []
    for k in d1.keys():
        if k in d2:
            lst.append(k)
    return lst

print(common_keys_yours({"a": 1, "c": 3}, {"c": 0, "a": 9}))   # → ['a', 'c']   looks right
print(common_keys_yours({"c": 3, "a": 1}, {"c": 0, "a": 9}))   # → ['c', 'a']   not sorted
print(sorted(k for k in {"c": 3, "a": 1} if k in {"c": 0, "a": 9}))   # → ['a', 'c']

Same lesson as the `chunk` bug: after a PASS, try one input of your own that is *different* from the test's. I've made Week 2's tests shuffle their inputs.

**3. When the input is wrong, `raise` an error instead of returning a message.** Your `matrix_add` returns the string `"error the matrix size must match"`. The caller might not notice, and later code would try to use that string as a matrix. Raising stops the program right at the real problem (Classes §7):

In [ ]:
def matrix_add(a, b):
    if len(a) != len(b) or len(a[0]) != len(b[0]):
        raise ValueError(f"shapes differ: {len(a)}x{len(a[0])} vs {len(b)}x{len(b[0])}")
    return [[a[i][j] + b[i][j] for j in range(len(a[0]))] for i in range(len(a))]

try:
    matrix_add([[1, 2]], [[1, 2, 3]])
except ValueError as e:
    print("Error:", e)          # → Error: shapes differ: 1x2 vs 1x3

**4. Your question: "`make_grid` just teaches how `.reshape` works?"** Partly, but the formula is the real point. Entry `[i, j]` holds `i * n + j`, which is **exactly the position of that entry when the grid is stored as one long row**. NumPy stores every array this way ("row-major" order), and that's why `reshape` costs nothing: it doesn't move any numbers, it only changes how they're indexed. You'll use this when a $28 \times 28$ image becomes a vector of $784$ numbers for a neural network (Week 5).

In [ ]:
import numpy as np
g = np.arange(9).reshape(3, 3)
print(g[1, 2], 1 * 3 + 2)        # → 5 5      [i, j] lives at position i*n + j in the flat array
img = np.zeros((28, 28))
print(img.reshape(-1).shape)     # → (784,)

**What "stored as one long row" means**

A NumPy array doesn't really store a grid. In memory, all the numbers sit **in one single line**, one after another, and the 2-D shape is just a way of reading that line.

You see this:

```text
          col 0   col 1   col 2
row 0  [    a       b       c   ]
row 1  [    d       e       f   ]
row 2  [    g       h       i   ]
```

But in memory, NumPy lays the rows out end to end, row 0 first, then row 1, then row 2:

```text
position:  0   1   2   3   4   5   6   7   8
value:     a   b   c   d   e   f   g   h   i
           └─ row 0 ─┘ └─ row 1 ─┘ └─ row 2 ─┘
```

This is **row-major** order: finish one row before starting the next.

**Finding a cell's position in that line**

Take `[2, 1]`, which holds `h`:

- Rows 0 and 1 come before it in the line. That's 2 full rows of 3 values each, so $2 \times 3 = 6$ values.
- Within row 2, there's 1 value (`g`) to its left.
- So `h` sits at position $6 + 1 = 7$.

In general, for a grid with $n$ columns, entry `[i, j]` sits at position $i \cdot n + j$: skip $i$ full rows of $n$, then move $j$ along. **That's exactly the number `make_grid` puts in each cell**: the grid is labelling every cell with its own address in memory.

**Why that makes reshape free**

`reshape` never moves a number. It only changes how the same line is read. The six numbers `0 1 2 3 4 5` read in rows of 3, and then in rows of 2:

```text
memory:        0  1  2  3  4  5

reshape(2, 3): [0 1 2]          read 3 at a time
               [3 4 5]

reshape(3, 2): [0 1]            read 2 at a time
               [2 3]
               [4 5]
```

A $28 \times 28$ image works the same way: `img.reshape(-1)` reads the 784 numbers as one line. Pixel `[1, 0]` (first pixel of the second row) lands at $1 \cdot 28 + 0 = 28$, and the last pixel `[27, 27]` at $27 \cdot 28 + 27 = 783$.

In [ ]:
img = np.arange(28 * 28).reshape(28, 28)
print(img[1, 0], img[27, 27], img.reshape(-1)[28])    # → 28 783 28

**5. Your comments on `ndim` vs "dimension".** You're right that the words clash. In linear algebra, a vector with 4 numbers is **4-dimensional**. In NumPy, that same vector has `ndim == 1`, because `ndim` counts **how many indices you need to reach one number**, exactly as you wrote. Read `ndim` as "number of axes":

| Object | Linear algebra says | NumPy `shape` | NumPy `ndim` |
|---|---|---|---|
| vector $x \in \mathbb{R}^4$ | 4-dimensional vector | `(4,)` | 1 |
| matrix $A \in \mathbb{R}^{2 \times 3}$ | $2 \times 3$ matrix | `(2, 3)` | 2 |
| a batch of 10 RGB images, $32 \times 32$ | (no single word) | `(10, 3, 32, 32)` | 4 |

So `shape` is the closest match to the linear-algebra size, as your other comment says. Notebook `03_shapes_and_slicing` works through this, since your quiz log flagged slicing and shapes.

**6. Small things.**

| Where | What you wrote | Shorter or safer |
|---|---|---|
| `Stack.is_empty` | `if not self.items: return True` / `return False` | `return not self.items` (the Week 0 lesson: return the condition itself) |
| `softmax` | computes `np.exp(z - z.max())` twice | `e = np.exp(z - z.max())` then `return e / e.sum()`: half the work, easier to read |
| `count_pairs` | builds a list of pairs, then `len()` | `count += 1` is enough when you only need the number |
| warm-up `chunk_yours` | fixed with `k`, but the old comment still says `# → [[1, 2], [4, 5]]` | update comments when the code changes. A stale comment misleads you later. |

**What you did well:** you found the shorter `count_above` yourself, and explained *why* `.sum()` needs no reshape. Your `__repr__` trick, `f"Rectangle{(self.width, self.height)}"`, is clever and correct. And your note "you don't pass `self` to `super().__init__`" is exactly right.

## Recap 1 · before Day 1 (Weeks 0–1)

**Quick check:** `d = {"a": 1}`. What is `d.get("b", 0) + d["a"]`?

<details><summary>Answer</summary>

`1`. `.get` returns the default 0 because `"b"` is missing, and `d["a"]` is 1.

</details>

**Quick check:** How many times does the inner line run?  `for i in range(4):` / `    for j in range(i, 4):` / `        ...`

<details><summary>Answer</summary>

$4 + 3 + 2 + 1 = 10$. For `i = 0` the inner loop runs over `j = 0..3` (4 times), for `i = 1` over `1..3` (3 times), and so on.

</details>

**Quick check:** `a = np.arange(1, 10).reshape(3, 3)`. What are `a[:, 0]` and its shape?

<details><summary>Answer</summary>

`[1 4 7]`, shape `(3,)`. `:` means all rows, and `0` means column 0, so you get the first number of every row.

</details>

**Quick check:** $P(A) = 0.3$, $P(B \mid A) = 0.5$, $P(B \mid \lnot A) = 0.2$. Find $P(A \mid B)$.

<details><summary>Answer</summary>

$P(A \cap B) = 0.5 \times 0.3 = 0.15$. $P(\lnot A \cap B) = 0.2 \times 0.7 = 0.14$. $P(A \mid B) = \frac{0.15}{0.15 + 0.14} = \frac{0.15}{0.29} \approx 0.517$.

</details>

**Quick check:** What does `super().__init__(side, side)` return?

<details><summary>Answer</summary>

`None`. `__init__` sets up the object and returns nothing, so don't assign its result to anything.

</details>

## Recap 2 · before Day 2

**Quick check:** `a.shape` is `(5, 3)`. What is the shape of `a.sum(axis=0)`?

<details><summary>Answer</summary>

`(3,)`. Axis 0 (the rows) is collapsed, leaving one total per column.

</details>

**Quick check:** $\operatorname{Var}(X) = 4$. What is $\operatorname{Var}(2X + 3)$?

<details><summary>Answer</summary>

$2^2 \times 4 = 16$. The $+3$ shifts the values but doesn't spread them.

</details>

**Quick check:** 9 successes in 12 trials. What is the MLE $\hat p$?

<details><summary>Answer</summary>

$\hat p = \frac{9}{12} = 0.75$.

</details>

**Quick check:** What is `[x * 2 for x in range(5) if x % 2 == 1]`?

<details><summary>Answer</summary>

`[2, 6]`. The odd values of `x` are 1 and 3, each doubled.

</details>

**Quick check:** $A$ is $2 \times 3$ and $B$ is $3 \times 4$. What is the shape of $AB$? And of $BA$?

<details><summary>Answer</summary>

$AB$ is $2 \times 4$. $BA$ is undefined: the inner sizes 4 and 2 don't match.

</details>

## Recap 3 · before Day 3

**Quick check:** $L(w) = (w - 2)^2$, $w = 0$, $\eta = 0.25$. One gradient step gives $w = $ ?

<details><summary>Answer</summary>

$L'(0) = 2(0 - 2) = -4$, so $w = 0 - 0.25 \times (-4) = 1$.

</details>

**Quick check:** Eigenvalues of $\begin{pmatrix} 2 & 0 \\ 0 & 5 \end{pmatrix}$?

<details><summary>Answer</summary>

$2$ and $5$: for a diagonal matrix, the eigenvalues are the diagonal entries.

</details>

**Quick check:** `a` is $3 \times 3$. Shapes of `a[:, 0]` and `a[:, 0:1]`?

<details><summary>Answer</summary>

`(3,)` and `(3, 1)`. An integer index drops that axis, while a slice keeps it, even a slice of length 1.

</details>

**Quick check:** The model gives the true class probability $0.25$. What is the cross-entropy loss?

<details><summary>Answer</summary>

$-\ln 0.25 = \ln 4 \approx 1.386$.

</details>

**Quick check:** In `Stack`, why does `self.items = []` go inside `__init__`?

<details><summary>Answer</summary>

So that **each** stack gets its own new list. A shared default list would be shared by every stack (the Week 0 mutable-default trap).

</details>

## Recap 4 · before Day 4 (now including this week)

**Quick check:** $u = (1, 2)$, $v = (3, -1)$. What is $u \cdot v$, and is the angle between them more or less than $90°$?

<details><summary>Answer</summary>

$1 \cdot 3 + 2 \cdot (-1) = 1 > 0$, so less than $90°$.

</details>

**Quick check:** Prior $\text{Beta}(3, 3)$, data: 2 heads in 4 flips. What is the MAP estimate?

<details><summary>Answer</summary>

$\hat p = \frac{h + a - 1}{n + a + b - 2} = \frac{2 + 2}{4 + 4} = 0.5$.

</details>

**Quick check:** $(AB)^\top = $ ?

<details><summary>Answer</summary>

$B^\top A^\top$: the order flips.

</details>

**Quick check:** $f(0) = 1$, $f(1) = 5$, $f(2) = 3$. What are $\max_\theta f(\theta)$ and $\arg\max_\theta f(\theta)$?

<details><summary>Answer</summary>

$\max = 5$ (the biggest **value**) and $\arg\max = 1$ (the **input** that gives it).

</details>

**Quick check:** Shape of `np.ones((4, 1)) + np.ones(3)`?

<details><summary>Answer</summary>

`(4, 3)`. From the right: 1 vs 3, so the 1 stretches. Then 4 vs nothing is fine.

</details>

## Recap 5 · before Day 5

**Quick check:** What is $D_{\mathrm{KL}}(p \,\Vert\, q)$ when $q = p$? Is $D_{\mathrm{KL}}$ symmetric?

<details><summary>Answer</summary>

$0$. No: in general $D_{\mathrm{KL}}(p \Vert q) \ne D_{\mathrm{KL}}(q \Vert p)$.

</details>

**Quick check:** What happens to ridge weights as $\lambda \to \infty$? As $\lambda \to 0$?

<details><summary>Answer</summary>

As $\lambda \to \infty$ they shrink toward $0$. As $\lambda \to 0$ ridge becomes ordinary least squares.

</details>

**Quick check:** Points $(0, 1), (1, 3), (2, 5)$. What is the least-squares line?

<details><summary>Answer</summary>

$\hat y = 2x + 1$. The points lie exactly on it, so the residuals are all 0.

</details>

**Quick check:** Project $b = (3, 4)$ onto $a = (1, 0)$.

<details><summary>Answer</summary>

$\frac{a \cdot b}{a \cdot a} a = \frac{3}{1}(1, 0) = (3, 0)$.

</details>

**Quick check:** `x = np.zeros((2, 3, 4))`. What are `x.ndim` and `x[0].shape`?

<details><summary>Answer</summary>

`3` and `(3, 4)`.

</details>

## Recap 6 · before Day 6

**Quick check:** $\sigma(0) = $ ?  And $\sigma(-z) = $ ?

<details><summary>Answer</summary>

$\sigma(0) = 0.5$ and $\sigma(-z) = 1 - \sigma(z)$.

</details>

**Quick check:** Binary cross-entropy $L$, with $p = \sigma(z)$. What is $\frac{\partial L}{\partial z}$?

<details><summary>Answer</summary>

$p - y$.

</details>

**Quick check:** $\text{SSE} = 2$ and $\text{SST} = 8$. What is $R^2$?

<details><summary>Answer</summary>

$R^2 = 1 - \frac{2}{8} = 0.75$.

</details>

**Quick check:** What is the entropy of a fair 4-sided die, in bits?

<details><summary>Answer</summary>

$\log_2 4 = 2$ bits.

</details>

**Quick check:** Minimizing which loss is the same as MLE under Gaussian noise?

<details><summary>Answer</summary>

Mean squared error.

</details>

## Recap 7 · before Day 7

**Quick check:** $TP = 8$, $FP = 2$, $FN = 8$. What are precision and recall?

<details><summary>Answer</summary>

Precision $= \frac{8}{8 + 2} = 0.8$. Recall $= \frac{8}{8 + 8} = 0.5$.

</details>

**Quick check:** Why must `StandardScaler` be fitted on the training set only?

<details><summary>Answer</summary>

Fitting on all the data lets the test set's mean and spread leak into training (data leakage), so the test score looks better than it really is.

</details>

**Quick check:** 1-NN on the 1-D training set $\{1 \to A,\ 4 \to B,\ 6 \to B\}$. What is the prediction for $x = 2$?

<details><summary>Answer</summary>

$A$. The distances are $1$, $2$, $4$, and the closest point (at 1) is $A$.

</details>

**Quick check:** Write the normal equation.

<details><summary>Answer</summary>

$X^\top X\, w = X^\top y$.

</details>

**Quick check:** `a = np.arange(12).reshape(3, 4)`. What are `a[1].shape` and `a[:, 1]`?

<details><summary>Answer</summary>

`(4,)` and `[1 5 9]`.

</details>

---
Use one recap set per day, before anything else. If you miss an answer, write it in that day's log entry: that topic goes into next week's recap.